In [ ]:
# ============================================================
# CELL 1: INSTALL REQUIRED LIBRARIES
# Mục đích:
# - Cài các thư viện cần thiết cho experiment KNN-OOD
# - PyTorch: chạy model
# - Torchvision: CIFAR-10
# - scikit-learn: tính khoảng cách / đánh giá sau này
# ============================================================

!pip install -q torch torchvision scikit-learn

In [ ]:
# ============================================================
# CELL 2: IMPORT LIBRARIES
# Mục đích:
# - Import PyTorch
# - Import Torchvision để làm việc với CIFAR-10
# - Import các công cụ xử lý dữ liệu
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import numpy as np

print("PyTorch version:", torch.__version__)
print("Libraries imported successfully!")

PyTorch version: 2.11.0+cpu
Libraries imported successfully!


In [ ]:
# ============================================================
# CELL 3: DEFINE IMAGE PREPROCESSING
# Mục đích:
# - Chuyển ảnh CIFAR-10 thành Tensor
# - Normalize theo chuẩn CIFAR-10
# - Dùng cùng preprocessing cho train/test/OOD
# ============================================================

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

print("Image preprocessing defined successfully!")

Image preprocessing defined successfully!


In [ ]:
# ============================================================
# CELL 4: LOAD CIFAR-10 TRAINING DATA
# Mục đích:
# - Tải CIFAR-10 training set
# - 50,000 ảnh
# - Dùng làm dữ liệu ID reference cho KNN
# ============================================================

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

print("Number of training images:", len(train_dataset))
print("Number of classes:", len(train_dataset.classes))
print("Classes:", train_dataset.classes)

100%|██████████| 170M/170M [01:40<00:00, 1.70MB/s]


Number of training images: 50000
Number of classes: 10
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [ ]:
# ============================================================
# CELL 5: CREATE TRAIN DATALOADER
# Mục đích:
# - Tạo DataLoader cho CIFAR-10 training set
# - Batch size = 128 để trích xuất feature hiệu quả
# - shuffle=False để giữ thứ tự dữ liệu cố định
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

print("Number of batches:", len(train_loader))
print("Batch size:", train_loader.batch_size)

Number of batches: 391
Batch size: 128


In [ ]:
# ============================================================
# CELL 6: LOAD PRETRAINED WIDE RESNET-40-2
# Mục đích:
# - Load đúng WideResNet-40-2 đã dùng ở experiment MSP
# - Model đã được pretrained trên CIFAR-10
# - Không train lại model
# ============================================================

import torch
import torch.nn as nn

# Download checkpoint
checkpoint_url = (
    "https://github.com/wetliu/energy_ood/raw/master/"
    "CIFAR/snapshots/pretrained/cifar10_wrn_pretrained_epoch_99.pt"
)

checkpoint_path = torch.hub.download_url_to_file(
    checkpoint_url,
    "/content/cifar10_wrn_pretrained_epoch_99.pt"
)

print("Checkpoint downloaded successfully!")

100%|██████████| 8.62M/8.62M [00:00<00:00, 146MB/s]

Checkpoint downloaded successfully!


In [ ]:
# ============================================================
# CELL 7: LOAD PRETRAINED MODEL
# Mục đích:
# - Load WideResNet-40-2 pretrained trên CIFAR-10
# - Đây là model dùng cho KNN
# - Không train lại model
# ============================================================

!pip install -q pytorch-ood

from pytorch_ood.model import load_model

model = load_model(
    "wrn-40-2/cifar10/crossentropy"
)

model.eval()

print("Model loaded successfully!")
print("Model:", type(model).__name__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.2/473.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 52.7 MB/s eta 0:00:00
Downloading: "https://github.com/wetliu/energy_ood/raw/master/CIFAR/snapshots/pretrained/cifar10_wrn_pretrained_epoch_99.pt" to /root/.cache/torch/hub/checkpoints/wrn-40-2-cifar10-crossentropy-0b59e178577382a0.pt


100%|██████████| 8.62M/8.62M [00:00<00:00, 62.0MB/s]

Model loaded successfully!
Model: WideResNet


In [ ]:
# ============================================================
# CELL 8: TEST MODEL
# Mục đích:
# - Kiểm tra model hoạt động bình thường
# - Đưa 1 batch CIFAR-10 vào model
# - Kiểm tra input và output
# - CHƯA chạy KNN
# ============================================================

images, labels = next(iter(train_loader))

with torch.no_grad():
    logits = model(images)

predictions = logits.argmax(dim=1)

print("Input shape :", images.shape)
print("Logits shape:", logits.shape)
print("Predictions :", predictions[:10].tolist())
print("True labels :", labels[:10].tolist())

Input shape : torch.Size([128, 3, 32, 32])
Logits shape: torch.Size([128, 10])
Predictions : [6, 9, 9, 4, 1, 1, 2, 7, 8, 3]
True labels : [6, 9, 9, 4, 1, 1, 2, 7, 8, 3]


In [ ]:
# ============================================================
# CELL 9: EXTRACT 128-D FEATURES
# Mục đích:
# - Lấy feature bên trong WideResNet
# - Mỗi ảnh sẽ được biểu diễn bằng 128 số
# - Kiểm tra kích thước feature trước khi chạy toàn bộ dataset
# - CHƯA chạy KNN
# ============================================================

with torch.no_grad():
    features = model.features(images)

print("Input shape    :", images.shape)
print("Feature shape  :", features.shape)

Input shape    : torch.Size([128, 3, 32, 32])
Feature shape  : torch.Size([128, 128])


In [ ]:
# ============================================================
# CELL 10: BUILD KNN REFERENCE FEATURES
# Mục đích:
# - Chạy toàn bộ CIFAR-10 TRAIN qua model
# - Lấy feature 128 chiều cho mỗi ảnh
# - Tạo feature bank để KNN sử dụng
# - Chưa chạy OOD
# ============================================================

import torch

all_features = []
all_labels = []

model.eval()

with torch.no_grad():

    for batch_idx, (images, labels) in enumerate(train_loader):

        features = model.features(images)

        all_features.append(features)
        all_labels.append(labels)

        if (batch_idx + 1) % 50 == 0:
            print(
                f"Processed {batch_idx + 1}/{len(train_loader)} batches"
            )

# Gộp tất cả batch lại
feature_bank = torch.cat(all_features, dim=0)
label_bank = torch.cat(all_labels, dim=0)

print("\n========== FEATURE BANK ==========")
print("Feature bank shape:", feature_bank.shape)
print("Label bank shape  :", label_bank.shape)

Processed 50/391 batches
Processed 100/391 batches
Processed 150/391 batches
Processed 200/391 batches
Processed 250/391 batches
Processed 300/391 batches
Processed 350/391 batches

========== FEATURE BANK ==========
Feature bank shape: torch.Size([50000, 128])
Label bank shape  : torch.Size([50000])


In [ ]:
# ============================================================
# CELL 11: LOAD CIFAR-10 TEST SET
# Mục đích:
# - Load CIFAR-10 test set
# - 10,000 ảnh
# - Đây là ID data dùng để đánh giá KNN
# - Chưa tính KNN
# ============================================================

cifar10_test = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

cifar10_test_loader = DataLoader(
    cifar10_test,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

print("CIFAR-10 test images:", len(cifar10_test))
print("Number of batches:", len(cifar10_test_loader))

CIFAR-10 test images: 10000
Number of batches: 79


In [ ]:
# ============================================================
# CELL 12: LOAD CIFAR-100 TEST SET
# Mục đích:
# - Load CIFAR-100 test set
# - 10,000 ảnh
# - CIFAR-100 là OOD đối với model CIFAR-10
# - Chưa tính KNN
# ============================================================

cifar100_test = datasets.CIFAR100(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

cifar100_test_loader = DataLoader(
    cifar100_test,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

print("CIFAR-100 test images:", len(cifar100_test))
print("Number of batches:", len(cifar100_test_loader))

100%|██████████| 169M/169M [00:31<00:00, 5.39MB/s]


CIFAR-100 test images: 10000
Number of batches: 79


In [ ]:
# ============================================================
# CELL 13: EXTRACT FEATURES FROM CIFAR-10 TEST
# Mục đích:
# - Đưa 10,000 ảnh CIFAR-10 test qua model
# - Lấy feature 128 chiều cho mỗi ảnh
# - Đây là feature của ID data
# - Chưa chạy KNN
# ============================================================

id_features = []
id_labels = []

model.eval()

with torch.no_grad():

    for batch_idx, (images, labels) in enumerate(cifar10_test_loader):

        features = model.features(images)

        id_features.append(features)
        id_labels.append(labels)

        if (batch_idx + 1) % 20 == 0:
            print(
                f"Processed {batch_idx + 1}/{len(cifar10_test_loader)} batches"
            )

# Gộp tất cả batch
id_features = torch.cat(id_features, dim=0)
id_labels = torch.cat(id_labels, dim=0)

print("\n========== CIFAR-10 TEST FEATURES ==========")
print("Feature shape:", id_features.shape)
print("Label shape  :", id_labels.shape)

Processed 20/79 batches
Processed 40/79 batches
Processed 60/79 batches

========== CIFAR-10 TEST FEATURES ==========
Feature shape: torch.Size([10000, 128])
Label shape  : torch.Size([10000])


In [ ]:
# ============================================================
# CELL 14: EXTRACT FEATURES FROM CIFAR-100 TEST
# Mục đích:
# - Đưa 10,000 ảnh CIFAR-100 test qua model
# - Lấy feature 128 chiều cho mỗi ảnh
# - Đây là feature của OOD data
# - Chưa chạy KNN
# ============================================================

ood_features = []
ood_labels = []

model.eval()

with torch.no_grad():

    for batch_idx, (images, labels) in enumerate(cifar100_test_loader):

        features = model.features(images)

        ood_features.append(features)
        ood_labels.append(labels)

        if (batch_idx + 1) % 20 == 0:
            print(
                f"Processed {batch_idx + 1}/{len(cifar100_test_loader)} batches"
            )

# Gộp tất cả batch
ood_features = torch.cat(ood_features, dim=0)
ood_labels = torch.cat(ood_labels, dim=0)

print("\n========== CIFAR-100 TEST FEATURES ==========")
print("Feature shape:", ood_features.shape)
print("Label shape  :", ood_labels.shape)

Processed 20/79 batches
Processed 40/79 batches
Processed 60/79 batches

========== CIFAR-100 TEST FEATURES ==========
Feature shape: torch.Size([10000, 128])
Label shape  : torch.Size([10000])


In [ ]:
# ============================================================
# CELL 15: TEST KNN ON ONE ID IMAGE(kiểu 1)
# Mục đích:
# - Lấy 1 ảnh từ CIFAR-10 TEST
# - So sánh với 50,000 feature của CIFAR-10 TRAIN
# - Tìm 10 hàng xóm gần nhất
# - Kiểm tra KNN hoạt động đúng
# - Chưa chạy toàn bộ 10,000 ảnh
# ============================================================

import torch

# Lấy feature của 1 ảnh CIFAR-10 TEST
query = id_features[0].unsqueeze(0)

# Tính khoảng cách Euclidean tới 50,000 feature trong feature bank
distances = torch.cdist(
    query,
    feature_bank
).squeeze(0)

# Chọn 10 hàng xóm gần nhất
K = 10

knn_distances, knn_indices = torch.topk(
    distances,
    k=K,
    largest=False
)

print("Query feature shape:", query.shape)
print("Number of distances:", distances.shape)

print("\nK nearest distances:")
print(knn_distances)

print("\nK nearest indices:")
print(knn_indices)

print("\nK nearest labels:")
print(label_bank[knn_indices])

Query feature shape: torch.Size([1, 128])
Number of distances: torch.Size([50000])

K nearest distances:
tensor([1.0901, 1.1038, 1.2558, 1.2678, 1.2851, 1.3578, 1.3687, 1.3759, 1.3849,
        1.3883])

K nearest indices:
tensor([41271, 10329, 23658, 29001, 23992, 44859,  8978, 17124, 23700,    36])

K nearest labels:
tensor([3, 3, 3, 3, 3, 3, 3, 3, 3, 3])


In [ ]:
# ============================================================
# CELL 16: TEST KNN ON ONE OOD IMAGE (kiểu 1)
# Mục đích:
# - Lấy 1 ảnh CIFAR-100 TEST
# - So sánh với 50,000 feature CIFAR-10 TRAIN
# - Tìm 10 hàng xóm gần nhất
# - So sánh khoảng cách với ảnh ID ở Cell 15
# - Chưa chạy toàn bộ OOD dataset
# ============================================================

# Lấy feature của 1 ảnh CIFAR-100 TEST
query_ood = ood_features[0].unsqueeze(0)

# Tính khoảng cách tới toàn bộ 50,000 feature CIFAR-10 TRAIN
ood_distances = torch.cdist(
    query_ood,
    feature_bank
).squeeze(0)

# Lấy 10 hàng xóm gần nhất
K = 10

ood_knn_distances, ood_knn_indices = torch.topk(
    ood_distances,
    k=K,
    largest=False
)

print("Query OOD feature shape:", query_ood.shape)
print("Number of distances:", ood_distances.shape)

print("\nK nearest distances:")
print(ood_knn_distances)

print("\nK nearest indices:")
print(ood_knn_indices)

print("\nK nearest labels:")
print(label_bank[ood_knn_indices])

Query OOD feature shape: torch.Size([1, 128])
Number of distances: torch.Size([50000])

K nearest distances:
tensor([1.4812, 1.4945, 1.5382, 1.5526, 1.5706, 1.5750, 1.5906, 1.5963, 1.6025,
        1.6053])

K nearest indices:
tensor([48370, 49020,  8634,  4008, 36441, 35597, 21637, 33152, 15788, 42185])

K nearest labels:
tensor([2, 2, 2, 2, 2, 2, 2, 2, 2, 2])


In [ ]:
# ============================================================
# CELL 15: TEST KNN ON ONE ID IMAGE
# Mục đích:
# - Lấy 1 ảnh CIFAR-10 TEST
# - So sánh với 50,000 CIFAR-10 TRAIN features
# - K = 50
# - Lấy khoảng cách tới hàng xóm thứ 50 làm KNN score
# ============================================================

K = 50

# Lấy 1 ảnh ID
query_id = id_features[0].unsqueeze(0)

# Tính khoảng cách tới toàn bộ 50,000 feature TRAIN
distances_id = torch.cdist(
    query_id,
    feature_bank
).squeeze(0)

# Lấy 50 hàng xóm gần nhất
knn_distances_id, knn_indices_id = torch.topk(
    distances_id,
    k=K,
    largest=False
)

# KNN score = khoảng cách tới hàng xóm thứ 50
id_score = knn_distances_id[-1]

print("Query feature shape:", query_id.shape)
print("Number of distances:", distances_id.shape)

print("\n50 nearest distances:")
print(knn_distances_id)

print("\n50 nearest indices:")
print(knn_indices_id)

print("\n50 nearest labels:")
print(label_bank[knn_indices_id])

print("\nKNN score (distance to 50th neighbor):",
      id_score.item())

Query feature shape: torch.Size([1, 128])
Number of distances: torch.Size([50000])

50 nearest distances:
tensor([1.0901, 1.1038, 1.2558, 1.2678, 1.2851, 1.3578, 1.3687, 1.3759, 1.3849,
        1.3883, 1.4513, 1.4548, 1.4746, 1.5065, 1.5274, 1.5573, 1.5611, 1.5809,
        1.5846, 1.5962, 1.6010, 1.6391, 1.6426, 1.6626, 1.6627, 1.6750, 1.6825,
        1.6882, 1.6920, 1.6921, 1.6962, 1.7027, 1.7059, 1.7096, 1.7099, 1.7102,
        1.7108, 1.7121, 1.7193, 1.7261, 1.7261, 1.7310, 1.7393, 1.7429, 1.7453,
        1.7472, 1.7488, 1.7624, 1.7628, 1.7660])

50 nearest indices:
tensor([41271, 10329, 23658, 29001, 23992, 44859,  8978, 17124, 23700,    36,
        36951, 39313, 14368, 11902, 27020, 22761, 23584, 45128, 39614,  6574,
        46590, 48966, 13093,  4720,   101, 36980, 45249, 48572, 45402, 16808,
        25673, 49444, 20961, 35581, 32142, 31449, 34661,  5733, 35923, 18482,
         7493, 22368, 23942, 15202, 38764, 41694, 46295, 13287, 36781, 41020])

50 nearest labels:
tensor([3, 3,

In [ ]:
# ============================================================
# CELL 16: TEST KNN ON ONE OOD IMAGE
# Mục đích:
# - Lấy 1 ảnh CIFAR-100 TEST
# - So sánh với 50,000 CIFAR-10 TRAIN features
# - K = 50
# - Lấy khoảng cách tới hàng xóm thứ 50 làm KNN score
# ============================================================

# Lấy 1 ảnh OOD
query_ood = ood_features[0].unsqueeze(0)

# Tính khoảng cách tới toàn bộ 50,000 feature TRAIN
distances_ood = torch.cdist(
    query_ood,
    feature_bank
).squeeze(0)

# Lấy 50 hàng xóm gần nhất
knn_distances_ood, knn_indices_ood = torch.topk(
    distances_ood,
    k=K,
    largest=False
)

# KNN score = khoảng cách tới hàng xóm thứ 50
ood_score = knn_distances_ood[-1]

print("Query OOD feature shape:", query_ood.shape)
print("Number of distances:", distances_ood.shape)

print("\n50 nearest distances:")
print(knn_distances_ood)

print("\n50 nearest indices:")
print(knn_indices_ood)

print("\n50 nearest labels:")
print(label_bank[knn_indices_ood])

print("\nKNN score (distance to 50th neighbor):",
      ood_score.item())

Query OOD feature shape: torch.Size([1, 128])
Number of distances: torch.Size([50000])

50 nearest distances:
tensor([1.4812, 1.4945, 1.5382, 1.5526, 1.5706, 1.5750, 1.5906, 1.5963, 1.6025,
        1.6053, 1.6068, 1.6085, 1.6294, 1.6475, 1.6531, 1.6601, 1.6677, 1.6751,
        1.6754, 1.6935, 1.6985, 1.6991, 1.7050, 1.7055, 1.7072, 1.7210, 1.7250,
        1.7307, 1.7337, 1.7358, 1.7378, 1.7467, 1.7481, 1.7518, 1.7528, 1.7582,
        1.7604, 1.7635, 1.7659, 1.7664, 1.7708, 1.7750, 1.7816, 1.7843, 1.7895,
        1.7898, 1.7926, 1.7968, 1.7985, 1.7990])

50 nearest indices:
tensor([48370, 49020,  8634,  4008, 36441, 35597, 21637, 33152, 15788, 42185,
        46183,  4634, 12978,  2138, 17712, 11380, 25192, 25449, 32687, 42601,
        40372, 29229,  5571, 17207,  2346,  8043, 20189, 22448, 43572, 25887,
         2205, 31041, 48344,  8237, 14811, 48702, 21063, 46943, 14840,  4588,
        39693, 16274, 39675,  2079, 19686, 13071, 20308, 28348, 49093, 15443])

50 nearest labels:
tensor([2

In [ ]:
# ============================================================
# CELL 17: COMPUTE KNN SCORES FOR ALL CIFAR-10 TEST IMAGES
# Mục đích:
# - Tính KNN score cho toàn bộ 10,000 ảnh ID
# - Reference: 50,000 CIFAR-10 TRAIN features
# - K = 50
# - Score = khoảng cách tới neighbor thứ 50
# ============================================================

K = 50
batch_size = 128

id_knn_scores = []

for start in range(0, len(id_features), batch_size):

    end = min(start + batch_size, len(id_features))

    query_batch = id_features[start:end]

    # Khoảng cách từ batch query tới 50,000 train features
    distances = torch.cdist(
        query_batch,
        feature_bank
    )

    # Lấy 50 hàng xóm gần nhất
    knn_distances, _ = torch.topk(
        distances,
        k=K,
        dim=1,
        largest=False
    )

    # KNN score = khoảng cách tới hàng xóm thứ 50
    scores = knn_distances[:, -1]

    id_knn_scores.append(scores)

    if end % 1000 == 0 or end == len(id_features):
        print(f"Processed {end}/{len(id_features)} ID images")

# Gộp score của toàn bộ 10,000 ảnh
id_knn_scores = torch.cat(id_knn_scores)

print("\n========== ID KNN SCORES ==========")
print("Number of scores:", len(id_knn_scores))
print("Mean   :", id_knn_scores.mean().item())
print("Median :", id_knn_scores.median().item())
print("Min    :", id_knn_scores.min().item())
print("Max    :", id_knn_scores.max().item())

Processed 10000/10000 ID images

========== ID KNN SCORES ==========
Number of scores: 10000
Mean   : 1.5406858921051025
Median : 1.418302297592163
Min    : 0.6654970645904541
Max    : 3.9033403396606445


In [ ]:
# ============================================================
# CELL 18: COMPUTE KNN SCORES FOR ALL CIFAR-100 TEST IMAGES
# Mục đích:
# - Tính KNN score cho toàn bộ 10,000 ảnh OOD
# - Reference: 50,000 CIFAR-10 TRAIN features
# - K = 50
# - Score = khoảng cách tới neighbor thứ 50
# ============================================================

K = 50
batch_size = 128

ood_knn_scores = []

for start in range(0, len(ood_features), batch_size):

    end = min(start + batch_size, len(ood_features))

    query_batch = ood_features[start:end]

    # Khoảng cách từ batch OOD tới 50,000 train features
    distances = torch.cdist(
        query_batch,
        feature_bank
    )

    # Lấy 50 hàng xóm gần nhất
    knn_distances, _ = torch.topk(
        distances,
        k=K,
        dim=1,
        largest=False
    )

    # KNN score = khoảng cách tới hàng xóm thứ 50
    scores = knn_distances[:, -1]

    ood_knn_scores.append(scores)

    if end % 1000 == 0 or end == len(ood_features):
        print(f"Processed {end}/{len(ood_features)} OOD images")

# Gộp score của toàn bộ 10,000 ảnh
ood_knn_scores = torch.cat(ood_knn_scores)

print("\n========== OOD KNN SCORES ==========")
print("Number of scores:", len(ood_knn_scores))
print("Mean   :", ood_knn_scores.mean().item())
print("Median :", ood_knn_scores.median().item())
print("Min    :", ood_knn_scores.min().item())
print("Max    :", ood_knn_scores.max().item())

Processed 10000/10000 OOD images

========== OOD KNN SCORES ==========
Number of scores: 10000
Mean   : 2.5213935375213623
Median : 2.571375608444214
Min    : 0.7883732914924622
Max    : 5.080453872680664


In [ ]:
# ============================================================
# CELL 19: COMPUTE KNN AUROC AND FPR95
# Mục đích:
# - Kết hợp 10,000 ID scores + 10,000 OOD scores
# - Tính AUROC
# - Tính FPR95
# - Score càng lớn = càng nghi ngờ OOD
# ============================================================

import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve

# Chuyển tensor -> NumPy
id_scores = id_knn_scores.cpu().numpy()
ood_scores = ood_knn_scores.cpu().numpy()

# Labels:
# 0 = ID
# 1 = OOD
y_true = np.concatenate([
    np.zeros(len(id_scores)),
    np.ones(len(ood_scores))
])

# Scores
y_scores = np.concatenate([
    id_scores,
    ood_scores
])

# ------------------------------------------------------------
# AUROC
# ------------------------------------------------------------

auroc = roc_auc_score(
    y_true,
    y_scores
)

# ------------------------------------------------------------
# FPR95
# FPR tại TPR >= 95%
# ------------------------------------------------------------

fpr, tpr, thresholds = roc_curve(
    y_true,
    y_scores
)

# Tìm điểm đầu tiên đạt TPR >= 95%
idx = np.where(tpr >= 0.95)[0][0]

fpr95 = fpr[idx]

print("========== KNN OOD DETECTION RESULTS ==========")
print(f"AUROC : {auroc:.4f} ({auroc * 100:.2f}%)")
print(f"FPR95 : {fpr95:.4f} ({fpr95 * 100:.2f}%)")

========== KNN OOD DETECTION RESULTS ==========
AUROC : 0.8895 (88.95%)
FPR95 : 0.4173 (41.73%)


In [ ]:
# ============================================================
# CELL 20: COMPUTE AUPR-IN AND AUPR-OUT
# Mục đích:
# - AUPR-IN  : ID là positive class
# - AUPR-OUT : OOD là positive class
# - KNN score càng lớn -> càng giống OOD
# ============================================================

from sklearn.metrics import average_precision_score
import numpy as np

# ------------------------------------------------------------
# AUPR-OUT
# OOD = positive
# KNN score càng lớn -> càng có khả năng OOD
# ------------------------------------------------------------

y_true_out = np.concatenate([
    np.zeros(len(id_scores)),     # ID = 0
    np.ones(len(ood_scores))      # OOD = 1
])

y_score_out = np.concatenate([
    id_scores,
    ood_scores
])

aupr_out = average_precision_score(
    y_true_out,
    y_score_out
)

# ------------------------------------------------------------
# AUPR-IN
# ID = positive
# KNN score càng nhỏ -> càng có khả năng ID
# Vì vậy đảo dấu KNN score
# ------------------------------------------------------------

y_true_in = np.concatenate([
    np.ones(len(id_scores)),      # ID = 1
    np.zeros(len(ood_scores))     # OOD = 0
])

y_score_in = np.concatenate([
    -id_scores,
    -ood_scores
])

aupr_in = average_precision_score(
    y_true_in,
    y_score_in
)

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("========== KNN AUPR RESULTS ==========")
print(f"AUPR-IN  : {aupr_in:.4f} ({aupr_in * 100:.2f}%)")
print(f"AUPR-OUT : {aupr_out:.4f} ({aupr_out * 100:.2f}%)")

========== KNN AUPR RESULTS ==========
AUPR-IN  : 0.8949 (89.49%)
AUPR-OUT : 0.8745 (87.45%)


In [ ]:
# ============================================================
# CELL 21: COMPUTE FULL PYTORCH-OOD METRICS
# Mục đích:
# - Dùng implementation chính thức của pytorch-ood
# - Tính AUROC
# - AUTC
# - AUPR-IN
# - AUPR-OUT
# - FPR95TPR
# ============================================================

from pytorch_ood.utils import OODMetrics

# ------------------------------------------------------------
# Gộp KNN scores
# ------------------------------------------------------------

all_scores = torch.tensor(
    np.concatenate([
        id_scores,
        ood_scores
    ]),
    dtype=torch.float32
)

# ------------------------------------------------------------
# Labels theo quy ước của pytorch-ood:
# 0  = ID / known
# -1 = OOD / unknown
# ------------------------------------------------------------

all_labels = torch.tensor(
    np.concatenate([
        np.zeros(len(id_scores)),       # ID
        -np.ones(len(ood_scores))       # OOD
    ]),
    dtype=torch.long
)

print("Scores shape :", all_scores.shape)
print("Labels shape:", all_labels.shape)
print("Unique labels:", torch.unique(all_labels))

# ------------------------------------------------------------
# OODMetrics
# ------------------------------------------------------------

metrics = OODMetrics()

metrics.update(
    all_scores,
    all_labels
)

results = metrics.compute()

print("\n========== KNN FULL METRICS ==========")

for name, value in results.items():
    print(f"{name:12s}: {value:.4f} ({value * 100:.2f}%)")

Scores shape : torch.Size([20000])
Labels shape: torch.Size([20000])
Unique labels: tensor([-1,  0])

========== KNN FULL METRICS ==========
AUROC       : 0.8895 (88.95%)
AUTC        : 0.3889 (38.89%)
AUPR-IN     : 0.8949 (89.49%)
AUPR-OUT    : 0.8745 (87.45%)
FPR95TPR    : 0.4173 (41.73%)


# **MNIST**

In [ ]:
# ============================================================
# CELL 22: LOAD MNIST TEST DATASET
# Mục đích:
# - Load toàn bộ 10,000 ảnh MNIST TEST
# - MNIST được sử dụng làm OOD dataset
# - Model vẫn là WideResNet-40-2 đã train trên CIFAR-10
# - Giữ preprocessing phù hợp với input của model CIFAR-10
# - Không thay đổi bất kỳ code nào của phần CIFAR-100
# ============================================================

mnist_test = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

mnist_test_loader = DataLoader(
    mnist_test,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

print("MNIST test images:", len(mnist_test))
print("Number of batches:", len(mnist_test_loader))

100%|██████████| 9.91M/9.91M [00:00<00:00, 16.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 747kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.63MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.66MB/s]

MNIST test images: 10000
Number of batches: 79


In [ ]:
# ============================================================
# CELL 22: LOAD MNIST TEST DATASET
# Mục đích:
# - Load toàn bộ 10,000 ảnh MNIST TEST
# - MNIST được sử dụng làm OOD dataset
# - Resize ảnh từ 28x28 -> 32x32
# - Chuyển ảnh grayscale 1 channel -> 3 channels
# - Sử dụng normalization của CIFAR-10
# - Không thay đổi bất kỳ code nào của phần CIFAR-100
# ============================================================

mnist_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

mnist_test = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=mnist_transform
)

mnist_test_loader = DataLoader(
    mnist_test,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

print("MNIST test images:", len(mnist_test))
print("Number of batches:", len(mnist_test_loader))

MNIST test images: 10000
Number of batches: 79


In [ ]:
# ============================================================
# CELL 23: EXTRACT MNIST FEATURES
# Mục đích:
# - Đưa toàn bộ MNIST test images qua WideResNet-40-2
# - Trích xuất feature vector từ model.features()
# - Mỗi ảnh được biểu diễn bằng vector 128 chiều
# - Các feature này sẽ được dùng cho KNN OOD detection
# ============================================================

mnist_features = []
mnist_labels = []

model.eval()

with torch.no_grad():
    for images, labels in mnist_test_loader:
        images = images.to(device)

        features = model.features(images)

        mnist_features.append(features.cpu())
        mnist_labels.append(labels)

mnist_features = torch.cat(mnist_features)
mnist_labels = torch.cat(mnist_labels)

print("MNIST feature shape:", mnist_features.shape)
print("MNIST label shape:", mnist_labels.shape)

MNIST feature shape: torch.Size([10000, 128])
MNIST label shape: torch.Size([10000])


In [ ]:
# ============================================================
# CELL 24: CALCULATE KNN SCORES FOR MNIST
# Mục đích:
# - So sánh feature MNIST với feature bank của CIFAR-10
# - Sử dụng K = 50 giống phần CIFAR-100
# - KNN score = khoảng cách tới neighbor thứ 50
# - Score càng cao -> càng khác CIFAR-10 -> càng có khả năng là OOD
# ============================================================

K = 50

mnist_knn_scores = []

for i in range(0, len(mnist_features), 128):
    batch_features = mnist_features[i:i+128]

    distances = torch.cdist(
        batch_features,
        feature_bank
    )

    knn_distances, _ = torch.topk(
        distances,
        k=K,
        dim=1,
        largest=False
    )

    scores = knn_distances[:, K-1]

    mnist_knn_scores.append(scores)

mnist_knn_scores = torch.cat(mnist_knn_scores)

print("MNIST KNN scores shape:", mnist_knn_scores.shape)
print("Mean:", mnist_knn_scores.mean().item())
print("Median:", mnist_knn_scores.median().item())
print("Min:", mnist_knn_scores.min().item())
print("Max:", mnist_knn_scores.max().item())

MNIST KNN scores shape: torch.Size([10000])
Mean: 2.688483715057373
Median: 2.7611896991729736
Min: 1.2114996910095215
Max: 3.610089063644409


In [ ]:
# ============================================================
# CELL 25: COMPUTE FULL PYTORCH-OOD METRICS FOR MNIST
# Mục đích:
# - Dùng implementation chính thức của pytorch-ood
# - CIFAR-10 test = ID
# - MNIST test = OOD
# - Giữ nguyên cách tính metrics như Cell 21
# - Tính AUROC
# - AUTC
# - AUPR-IN
# - AUPR-OUT
# - FPR95TPR
# ============================================================

# ------------------------------------------------------------
# Gộp KNN scores
# ------------------------------------------------------------

all_scores_mnist = torch.tensor(
    np.concatenate([
        id_scores,
        mnist_knn_scores.cpu().numpy()
    ]),
    dtype=torch.float32
)

# ------------------------------------------------------------
# Labels theo đúng quy ước của pytorch-ood:
# 0  = ID / known
# -1 = OOD / unknown
# ------------------------------------------------------------

all_labels_mnist = torch.tensor(
    np.concatenate([
        np.zeros(len(id_scores)),                    # ID
        -np.ones(len(mnist_knn_scores))              # OOD
    ]),
    dtype=torch.long
)

print("Scores shape :", all_scores_mnist.shape)
print("Labels shape:", all_labels_mnist.shape)
print("Unique labels:", torch.unique(all_labels_mnist))

# ------------------------------------------------------------
# OODMetrics
# ------------------------------------------------------------

metrics_mnist = OODMetrics()

metrics_mnist.update(
    all_scores_mnist,
    all_labels_mnist
)

results_mnist = metrics_mnist.compute()

print("\n========== KNN FULL METRICS - MNIST ==========")

for name, value in results_mnist.items():
    print(f"{name:12s}: {value:.4f} ({value * 100:.2f}%)")

Scores shape : torch.Size([20000])
Labels shape: torch.Size([20000])
Unique labels: tensor([-1,  0])

========== KNN FULL METRICS - MNIST ==========
AUROC       : 0.9373 (93.73%)
AUTC        : 0.3228 (32.28%)
AUPR-IN     : 0.9493 (94.93%)
AUPR-OUT    : 0.9130 (91.30%)
FPR95TPR    : 0.2246 (22.46%)
